# ZS601: controlled first-contributor depth
Same 10 virtual cameras, same no-glass 1 cm points, k=3, opacity=1 (float32 sigmoid20), no training.
Three variants: center Z scale0.5; ray peak Z scale0.5; ray peak Z scale0.1.
Peak is the front/back 3-sigma intersection midpoint when a real intersection exists; otherwise it remains the infinite Gaussian ray maximum. No silent discriminant gate.
First contributor uses the existing tile list sorted by Gaussian center Z; it is not exact entry-surface ordering. Mini-Splatting2 uses maximum alpha*T selection, so this is a controlled variant, not its full reproduction.


In [1]:
from pathlib import Path
import os,sys,json,subprocess,hashlib,platform,torch
ROOT=Path('/content/zs601-firsthit-depth-v008')
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
for row in json.loads((ROOT/'input_manifest.json').read_text()):assert sha(ROOT/row['path'])==row['sha256'],row['path']
env=dict(python=platform.python_version(),torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(),capability=torch.cuda.get_device_capability())
assert sys.version_info[:2]==(3,13) and env['torch']=='2.11.0+cu128' and env['capability']==(8,9)
(ROOT/'environment.json').write_text(json.dumps(env,indent=2))
print(env)


{'python': '3.13.15', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA L4', 'capability': (8, 9)}


In [2]:
def run(cmd,name,env=None):
 with (ROOT/name).open('x') as f:
  proc=subprocess.Popen(list(map(str,cmd)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env)
  for line in proc.stdout:f.write(line);f.flush();print(line,end='',flush=True)
  assert proc.wait()==0,name
run([sys.executable,'-m','pip','install','plyfile==1.1.3','ninja'],'install_python.log')
import zipfile
with zipfile.ZipFile(ROOT/'wheels.zip') as z:z.extractall(ROOT/'wheels')
knn=list((ROOT/'wheels').glob('simple_knn*.whl'));assert len(knn)==1
run([sys.executable,'-m','pip','install','--no-deps',knn[0]],'install_knn.log')
buildenv=os.environ.copy();buildenv['TORCH_CUDA_ARCH_LIST']='8.9';buildenv['MAX_JOBS']='2'
run([sys.executable,'-m','pip','install','--no-build-isolation','--no-deps',ROOT/'rasterizer'],'build_cuda.log',buildenv)
print('MODIFIED_CUDA_EXTENSION_BUILT')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.2 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 18.4 MB/s eta 0:00:00


Processing ./wheels/simple_knn-1.0.0-cp313-cp313-linux_x86_64.whl


Processing ./rasterizer


  Preparing metadata (setup.py): started


  Preparing metadata (setup.py): finished with status 'done'


  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp313-cp313-linux_x86_64.whl size=3492341 sha256=21cefe2a1188c277eb0d7359c9e9fb6fc03a4689a804e042845550a7d895435e


  Stored in directory: /tmp/pip-ephem-wheel-cache-1is4xlvd/wheels/f0/03/18/d25c60b5dc622444376a2763b6b84147ac634f553eadd02b43


Successfully built diff_gaussian_rasterization


MODIFIED_CUDA_EXTENSION_BUILT


In [3]:
run([sys.executable,'-u',ROOT/'run_first_hit.py','--package',ROOT/'package','--root',ROOT],'render.log')
print((ROOT/'results/analytic_tests.json').read_text())
print((ROOT/'results/regression_tests.json').read_text())
print((ROOT/'results/RENDER_COMPLETE.json').read_text())


{"method": "method_c_first_center_s05", "name": "003202.png", "mode": 1, "scale": 0.5, "coverage": 1.0, "alpha50_coverage": 1.0, "rgb_mask_gap": 1.1281588447653429e-05, "strictly_empty_alpha16": 0.0, "first_hit_without_real_3sigma_intersection_fraction": 0.7192167757220217, "first_alpha_median": 0.01052496861666441, "quantization_max_mm": 0.5002403259277699, "seconds": 1.891347885131836}


{"method": "method_c_first_center_s05", "name": "003340.png", "mode": 1, "scale": 0.5, "coverage": 1.0, "alpha50_coverage": 0.9999971796028881, "rgb_mask_gap": 0.00022986236462093861, "strictly_empty_alpha16": 0.0, "first_hit_without_real_3sigma_intersection_fraction": 0.6079774932310469, "first_alpha_median": 0.013424944132566452, "quantization_max_mm": 0.500118255615245, "seconds": 1.8276288509368896}


{"method": "method_c_first_center_s05", "name": "003376.png", "mode": 1, "scale": 0.5, "coverage": 1.0, "alpha50_coverage": 1.0, "rgb_mask_gap": 9.448330324909747e-05, "strictly_empty_alpha16": 0.0, "first_hit_without_real_3sigma_intersection_fraction": 0.6251212770758122, "first_alpha_median": 0.012939631938934326, "quantization_max_mm": 0.5000877380370028, "seconds": 1.8394851684570312}


{"method": "method_c_first_center_s05", "name": "003388.png", "mode": 1, "scale": 0.5, "coverage": 1.0, "alpha50_coverage": 1.0, "rgb_mask_gap": 0.0003624210288808664, "strictly_empty_alpha16": 0.0, "first_hit_without_real_3sigma_intersection_fraction": 0.5901582242779784, "first_alpha_median": 0.013629036955535412, "quantization_max_mm": 0.5001125335692969, "seconds": 1.799015760421753}


{"method": "method_c_first_center_s05", "name": "003409.png", "mode": 1, "scale": 0.5, "coverage": 1.0, "alpha50_coverage": 1.0, "rgb_mask_gap": 0.0032223037003610107, "strictly_empty_alpha16": 0.0, "first_hit_without_real_3sigma_intersection_fraction": 0.4423905685920578, "first_alpha_median": 0.017844777554273605, "quantization_max_mm": 0.5000495910645331, "seconds": 2.0233724117279053}


...

{"method": "method_e_first_peak_s01", "name": "003505.png", "mode": 2, "scale": 0.1, "coverage": 0.9919407152527075, "alpha50_coverage": 0.7987590252707581, "rgb_mask_gap": 0.5366129851083032, "strictly_empty_alpha16": 0.008059284747292419, "first_hit_without_real_3sigma_intersection_fraction": 0.8128432410915476, "first_alpha_median": 0.03896541893482208, "quantization_max_mm": 0.5002441406247726, "seconds": 1.5289857387542725}


{"method": "method_e_first_peak_s01", "name": "003520.png", "mode": 2, "scale": 0.1, "coverage": 0.9772224729241877, "alpha50_coverage": 0.6183466832129964, "rgb_mask_gap": 0.758462601534296, "strictly_empty_alpha16": 0.022777527075812275, "first_hit_without_real_3sigma_intersection_fraction": 0.7422189769224553, "first_alpha_median": 0.05170349031686783, "quantization_max_mm": 0.5001220703126918, "seconds": 1.52939772605896}


{"method": "method_e_first_peak_s01", "name": "003571.png", "mode": 2, "scale": 0.1, "coverage": 0.9500451263537906, "alpha50_coverage": 0.3071045803249097, "rgb_mask_gap": 0.965592565433213, "strictly_empty_alpha16": 0.04995487364620939, "first_hit_without_real_3sigma_intersection_fraction": 0.600530506341139, "first_alpha_median": 0.07049645483493805, "quantization_max_mm": 0.5000610351562074, "seconds": 1.5851590633392334}


{"method": "method_e_first_peak_s01", "name": "003583.png", "mode": 2, "scale": 0.1, "coverage": 0.9965196299638989, "alpha50_coverage": 0.7836388763537906, "rgb_mask_gap": 0.5293843073104693, "strictly_empty_alpha16": 0.003480370036101083, "first_hit_without_real_3sigma_intersection_fraction": 0.8114234446375302, "first_alpha_median": 0.036427564918994904, "quantization_max_mm": 0.5002212524409799, "seconds": 1.5270097255706787}


{
  "legacy_isolated": {
    "valid_pixels": 164,
    "max_depth_error_m": 1.1682510375976562e-05,
    "max_straight_rgb_error": 5.221366882302014e-06
  },
  "status": "PASS",
  "rgb_bit_exact_all_modes": true,
  "same_first_ids": true,
  "cpu_peak_max_error_m": 1.49391544468358e-07,
  "cpu_qmin_max_error": 2.3338397276972955e-06,
  "real_intersections_midpoint_max_error_m": 2.384185791015625e-07,
  "center_max_error_m": 0.0,
  "harmonic_background_mixing_max_m": 1.9135494232177734,
  "nonintersection_pixels": 244,
  "backward_guard": true
}

{
  "status": "PASS",
  "legacy": [
    {
      "name": "003202.png",
      "rgb_bit_exact": true,
      "legacy_depth_png_bit_exact": true
    },
    {
      "name": "003340.png",
      "rgb_bit_exact": true,
      "legacy_depth_png_bit_exact": true
    },
    {
      "name": "003376.png",
      "rgb_bit_exact": true,
      "legacy_depth_png_bit_exact": true
    },
    {
      "name": "003388.png",
      "rgb_bit_exact": true,
      "legacy_depth